# Où naît le flou ? — exécution sur Kaggle (GPU T4)

**Une session = un « Save & Run All (Commit) »** : il tourne en arrière-plan (fenêtre fermée, machine en veille), 12 h au plus.

L'état du projet (`runs/`, `reports/`) vit dans le Dataset privé **`flou-wm-etat`**. La section « 1. GPU, état sauvegardé et dépôt » le recopie dans
`/kaggle/working/flou_wm`, l'entraînement reprend au dernier checkpoint, puis `save_state()` publie une **nouvelle version**
du Dataset. Les données (`data/`) se régénèrent à l'identique à chaque session.

**Déroulé de la campagne** (variable `STAGE` de la section « 0. Réglages de la session »)
1. ✔ **Fait (session 1)** — `STAGE = 'first_run'` : `core-regression-p0.30-s0` terminé et évalué, feu vert (H0b 1,011,
   validation plate, flou 98,2 %). C'était le contrôle demandé après l'incident J6. On n'y revient plus.
2. ➜ **En cours : `STAGE = 'core_control'`**, les 12 runs prioritaires (cœur + contrôles). La session 2 en a évalué 4 et
   entamé le run de jetons s1. **Cette session (la 3e)** le termine, puis enchaîne régression s2, diffusion s2, jetons s2
   et oracle (environ 9 h), et entame les contrôles p = 0. La session suivante finit les deux contrôles p = 0.
3. Les mesures décidées le 17 novembre (J9) sont **allumées** (`RUN_REEVAL`, `RUN_SIGMA_SWEEP`) : environ 20 minutes
   avant l'entraînement. Les deux sont idempotentes, donc inoffensives d'une session à l'autre : les laisser allumées.
4. Quand les 12 runs sont évalués : `RUN_ANALYSIS = True`, puis l'extension « décision » si on la fait.

**Après chaque session**, une seule chose à régler : relever `MIN_EVALUATED` (section 0) au nombre de runs évalués que
la dernière ligne de la session affiche. C'est ce qui empêche de repartir d'un état périmé et de refaire des heures de GPU.

**Avant chaque session** : panneau de droite, accélérateur **GPU T4 x2**, **Internet On**, entrée **`flou-wm-etat`** sur sa
**dernière** version (accepter la mise à jour proposée), secret **`flou-wm`** coché (Add-ons → Secrets). Puis
**Save Version → Save & Run All (Commit)**. Le notebook refuse de démarrer si l'état repris compte moins de
`MIN_EVALUATED` runs évalués, et la section 1 affiche le commit cloné : il doit être le dernier de `main`.

Filet de sécurité : `/kaggle/working/flou_wm` devient aussi la sortie (*Output*) de la version ; si l'envoi au Dataset
échoue, les fichiers restent récupérables là.

La cellule du notebook Colab qui archivait le run à 100 000 transitions et vidait `runs/` n'existe pas ici : elle a déjà
servi (archive sur Drive) et la relancer effacerait le run en cours.

## 0. Réglages de la session

In [ ]:
# ===== Session settings: the only cell to edit between sessions =====
STAGE = 'core_control'     # 'first_run'    : DONE in session 1 (kept only to resume that run if ever needed)
                           # 'core_control' : the 12 priority runs (core + controls): the current stage
                           # 'none'         : no training (analysis or decision extension only)
FIRST_RUN = 'core-regression-p0.30-s0'
STEPS = 34800              # steps per run (plan, step 6, 38 h budget): do NOT change during the campaign
MAX_HOURS = 10             # Kaggle session: 12 h max -> keep ~2 h for evaluation and saving
                           # (session 2 lasted 10.05 h with this value; the 17 Nov measures add ~20 min before training)
MIN_EVALUATED = 5          # runs already evaluated in the state we expect to resume. After EACH session, set it to the
                           # count printed on the last line of the session (5 now). Guards against a stale Dataset input.
RUN_TESTS = False          # steps 2-3: already passed on Kaggle (all [OK]); set True again if the code changes
RUN_ANALYSIS = False       # step 8: once the 12 core + control runs are evaluated
RUN_REEVAL = True          # re-evaluate the finished runs that lack the rail-restricted measure (17 Nov, DECISIONS.md).
                           # Idempotent: a run that already carries it is skipped, so leave this on.
RUN_SIGMA_SWEEP = True     # one Euler step from several starting sigmas, diffusion runs only (J9).
                           # Idempotent: a run already swept is skipped, so leave this on.
RUN_DECISION_L1 = False    # step 9, level 1 + H0e/H0f controls: fast, after the analysis
RUN_DECISION_L2 = False    # step 9, level 2: LONG (read the corresponding cell before enabling it)

DATASET = 'cedricdada/flou-wm-etat'
SECRET_NAME = 'flou-wm'
MAX_SECONDS = int(MAX_HOURS * 3600)

## 1. GPU, état sauvegardé et dépôt

In [ ]:
!nvidia-smi -L
import os, glob, json, shutil, subprocess, time, zipfile
BASE_DIR = '/kaggle/working/flou_wm'
os.makedirs(BASE_DIR, exist_ok=True)


def restore_state(root):
    # Trouve le dossier qui contient runs/ sous `root` et le copie dans BASE_DIR.
    for current, dirs, files in os.walk(root):
        if 'runs' in dirs:
            shutil.copytree(current, BASE_DIR, dirs_exist_ok=True)
            return current
    return None


archives = glob.glob('/kaggle/input/**/etat.zip', recursive=True)
if archives:                                    # zip resté tel quel
    zipfile.ZipFile(archives[0]).extractall('/tmp/etat')
    state_source = restore_state('/tmp/etat')
else:                                           # Kaggle a décompressé le zip
    state_source = restore_state('/kaggle/input')
print("état repris depuis :", state_source or "RIEN TROUVÉ : départ à zéro (entrée flou-wm-etat ajoutée ?)")


def save_state(note):
    # Publie runs/ et reports/ comme nouvelle version du Dataset. N'interrompt jamais le notebook.
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(SECRET_NAME)
        os.environ['KAGGLE_USERNAME'] = DATASET.split('/')[0]
        os.environ['KAGGLE_KEY'] = token            # ancien format de clé
        os.environ['KAGGLE_API_TOKEN'] = token      # nouveau format (onglet « API Tokens »)
        upload_dir = '/tmp/up'
        shutil.rmtree(upload_dir, ignore_errors=True)
        os.makedirs(f'{upload_dir}/etat')
        for name in ('runs', 'reports'):
            if os.path.isdir(f'{BASE_DIR}/{name}'):
                shutil.copytree(f'{BASE_DIR}/{name}', f'{upload_dir}/etat/{name}')
        shutil.make_archive(f'{upload_dir}/etat', 'zip', f'{upload_dir}/etat')
        shutil.rmtree(f'{upload_dir}/etat')         # n'envoyer que etat.zip
        json.dump({"id": DATASET, "title": DATASET.split('/')[1], "licenses": [{"name": "other"}]},
                  open(f'{upload_dir}/dataset-metadata.json', 'w'))
        version_note = f"{note} ({time.strftime('%Y-%m-%d %H:%M')} UTC)"
        size_mb = os.path.getsize(f'{upload_dir}/etat.zip') / 1e6
        print(f"envoi de etat.zip ({size_mb:.0f} Mo) : {version_note}")
        result = subprocess.run(['kaggle', 'datasets', 'version', '-p', upload_dir, '-m', version_note],
                                capture_output=True, text=True)
        print(result.stdout[-1500:], result.stderr[-1500:])
        print("SAUVEGARDE OK" if result.returncode == 0 else
              f"ÉCHEC DE LA SAUVEGARDE (code {result.returncode}) : récupérer {BASE_DIR} dans l'Output de la version")
    except Exception as e:
        print(f"ÉCHEC DE LA SAUVEGARDE ({type(e).__name__}: {e}) : récupérer {BASE_DIR} dans l'Output de la version")


!ls {BASE_DIR}/runs/*/

# Garde-fou : l'état repris doit contenir au moins MIN_EVALUATED runs évalués. Tester le seul premier run ne suffit pas :
# une entrée restée sur la version de la session 1 le contient, et la session recommencerait des heures de GPU.
n_evaluated = len(glob.glob(f'{BASE_DIR}/runs/*/eval.json'))
print(f"runs évalués dans l'état repris : {n_evaluated} (au moins {MIN_EVALUATED} attendus)")
if STAGE == 'core_control' and n_evaluated < MIN_EVALUATED:
    raise RuntimeError(f"Entrée flou-wm-etat périmée : {n_evaluated} runs évalués, {MIN_EVALUATED} attendus. "
                       "Mettre l'entrée sur sa dernière version avant de lancer la session.")

In [ ]:
# Cloner le dépôt (dépôt privé : mettre un jeton -> https://<TOKEN>@github.com/...). Exige Internet On.
REPO_URL = 'https://github.com/CedricDada/flou-world-models.git'
REPO_DIR = '/content/flou_wm'
# Clone neuf à chaque session : c'est la seule façon sûre d'avoir le dernier code.
if os.path.isdir(REPO_DIR):
    %cd /content
    shutil.rmtree(REPO_DIR)
!git clone -q $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git log --oneline -1

# runs/ reports/ data/ du dépôt pointent vers BASE_DIR (qui part dans le Dataset et dans l'Output de la version)
for name in ('runs', 'reports', 'data'):
    os.makedirs(f'{BASE_DIR}/{name}', exist_ok=True)
    if os.path.isdir(name) and not os.path.islink(name):
        shutil.rmtree(name)
    if not os.path.islink(name):
        os.symlink(f'{BASE_DIR}/{name}', name)

# Kaggle fournit déjà numpy, scipy, matplotlib et pillow : n'installer que ce qui manque,
# pour ne pas risquer de remplacer la numpy sur laquelle le PyTorch préinstallé est compilé.
import importlib.util
missing = [package for module, package in (('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib'),
                                           ('PIL', 'pillow'), ('pytest', 'pytest'))
           if importlib.util.find_spec(module) is None]
if missing:
    !pip install -q {' '.join(missing)}
print('paquets installés :', ' '.join(missing) if missing else 'aucun, tout était présent')

## 2-3. Tests NumPy et test de fumée PyTorch (une fois, ou quand le code change)

In [ ]:
if RUN_TESTS:
    !python -m pytest -q -x
    !python scripts/smoke_test.py
else:
    print("tests sautés (RUN_TESTS = False)")

## 4. Données

Régénérées à chaque session (une minute environ), identiques d'une session à l'autre.

In [ ]:
!python scripts/generate_data.py --split train --p 0.0 0.1 0.3 0.5 --n-transitions 400000
!python scripts/generate_data.py --split val   --p 0.0 0.1 0.3 0.5 --n-transitions 20000
!python scripts/generate_data.py --split test  --p 0.0 0.1 0.3 0.5 --n-bifurcations 2000   # H2b exige >= 1 000 images NETTES par graine

## 5. Instrument de mesure (avant tout modèle)

Règle ρmax et contrôle H0a sans entraînement. Si H0a échoue, **ne mesurer aucun modèle**.

In [ ]:
!python scripts/validate_instrument.py

## 6bis. Mesures décidées le 17 novembre (J9) — AVANT l'entraînement

Les deux sont **exploratoires** : elles ne changent aucun verdict, et sont déclarées dans
`docs/DECISIONS.md` avant toute analyse. Les deux sont **idempotentes** : on peut les laisser allumées.

- `RUN_REEVAL` réévalue les runs terminés dont l'`eval.json` n'a pas encore la mesure restreinte au rail de la
  bifurcation (clé `rail`). Aujourd'hui, ce sont les 5 runs déjà évalués ; plus tard, aucun : `evaluate.py` écrit la
  clé de lui-même. Aucun réentraînement.
- `RUN_SIGMA_SWEEP` mesure un pas d'Euler depuis plusieurs σ de départ, sur les runs de diffusion pas encore
  balayés. Il dit *pourquoi* la diffusion à n = 1 donne 100 % d'images « autre », sans changer H2a. **Se lit contre la
  référence d'un débruiteur exact**, affichée juste avant (voir `DECISIONS.md`) : à σ petit, la bonne attente est
  des images *nettes*, pas floues.

**Pourquoi avant l'entraînement et non après.** La réévaluation coûte environ 14 minutes (mesurées sur les cinq runs :
diffusion 5,2 min, jetons 2,9 min, régression 0,3 min) ; le balayage quelques minutes de plus, non chronométrées.
L'entraînement, lui, s'arrête proprement à `MAX_HOURS` et reprend : c'est le seul des deux qu'on peut tronquer. Placées
après, ces minutes risqueraient de pousser la session au-delà des 12 h et de faire sauter la sauvegarde.

Les erreurs de ce bloc sont **attrapées** : une réévaluation qui échoue ne doit pas coûter les 10 heures qui suivent.

In [ ]:
# Réévaluation (idempotente) : seuls les runs finis dont l'eval.json n'a pas la clé `rail` sont refaits.
if RUN_REEVAL:
    try:
        finished = [os.path.basename(os.path.dirname(p)) for p in sorted(glob.glob('runs/*/eval.json'))]
        stale = [r for r in finished if os.path.exists(f'runs/{r}/final.pt') and not all(
            'rail' in v['summary'] for v in json.load(open(f'runs/{r}/eval.json'))['variants'])]
        print("à réévaluer (sans clé rail) :", stale or "aucun")
        for run_id in stale:
            !python scripts/evaluate.py --run-id {run_id} --force
    except Exception as e:
        print(f"réévaluation impossible ({type(e).__name__}: {e}) : la session continue")
else:
    print("réévaluation sautée (RUN_REEVAL = False)")

# Balayage de sigma (idempotent : un run déjà balayé est sauté). La référence exacte s'affiche d'abord.
if RUN_SIGMA_SWEEP:
    try:
        !python scripts/ideal_sigma_reference.py --p 0.3
        !python scripts/sigma_sweep.py --all --sigmas 1 2 5 10 20 30 40 60 80
    except Exception as e:
        print(f"balayage de sigma impossible ({type(e).__name__}: {e}) : la session continue")
else:
    print("balayage de sigma sauté (RUN_SIGMA_SWEEP = False)")

## 7. Entraînement

- `first_run` : `train.py` reprend `FIRST_RUN` au dernier checkpoint et le termine, puis `evaluate.py` l'évalue.
- `core_control` : `run_plan.py` enchaîne les 12 runs prioritaires (entraînement puis évaluation), reprend un run commencé
  au pas près et saute ceux qui sont finis.

Dans les deux cas, `--max-seconds` / `--max-hours` est un arrêt propre : le run en cours écrit son checkpoint et la session
suivante le reprend.

In [ ]:
!python scripts/run_plan.py --steps {STEPS} --groups core control --dry    # état avant la session
if STAGE == 'first_run':
    !python scripts/train.py --run-id {FIRST_RUN} --steps {STEPS} --max-seconds {MAX_SECONDS}
    if os.path.exists(f'runs/{FIRST_RUN}/final.pt'):
        !python scripts/evaluate.py --run-id {FIRST_RUN}
    else:
        print(f"{FIRST_RUN} pas encore fini : relancer une session pour reprendre")
elif STAGE == 'core_control':
    !python scripts/run_plan.py --steps {STEPS} --max-hours {MAX_HOURS} --groups core control
else:
    print("pas d'entraînement (STAGE = 'none')")

## Contrôle du surapprentissage (incident J6)

Deux symptômes sont cherchés dans les journaux (`scripts/training_curves.py`) : la perte de validation qui **remonte**
depuis son minimum, et la perte d'entraînement qui passe **sous le plancher de Bayes** p(1−p)δ² (impossible sans
mémoriser l'issue des bifurcations). Puis, sur le run évalué : **H0b** (MSE entre 1,00 et 1,10 fois le plancher, pas de
fuite) et le flou de la régression (côté régression de H1). Référence du run raté J6 (100 000 transitions) : MSE/plancher
1,47, flou 22,5 %, autre 2,0 %, p̂ = 0,158, perte d'entraînement 0,46 fois le plancher, validation +49 % après le pas 14 000.

In [ ]:
def report_run(run_id):
    # Résume eval.json : flou, autre, p̂, rappel du petit objet ; H0b pour la régression seulement.
    # Renvoie True ou False (H0b) pour un run de régression, None sinon.
    path = f'runs/{run_id}/eval.json'
    if not os.path.exists(path):
        print(f"{run_id} : pas encore évalué")
        return None
    evaluation = json.load(open(path))
    family = evaluation['spec']['family']
    h0b_passed = True
    for variant in evaluation['variants']:
        summary = variant['summary']
        n, counts, p_hat = summary['n'], summary['counts'], summary['p_hat']
        print(f"{run_id} {variant['variant']}  ({family}, ρmax = {summary['rho_max']:.4f}, n = {n} images)")
        if family == 'regression':      # H0b n'a de sens que pour la régression (l'oracle connaît la branche)
            h0b = variant['chk_h0b']
            print(f"  H0b : MSE / plancher de Bayes = {h0b['ratio_expected']:.3f} en espérance (attendu 1,00 à 1,10), "
                  f"{h0b['ratio_realized']:.3f} contre l'image tirée ; fuite z = {h0b['leak_z']:.1f} (alerte si > 3)"
                  f"  ->  {'RÉUSSI' if h0b['pass'] else 'ÉCHOUÉ'}")
            h0b_passed = h0b_passed and h0b['pass']
        print(f"  flou {counts['floue'] / n:.1%}   autre {counts['autre'] / n:.1%}"
              f"   nettes {(counts['nette-A'] + counts['nette-B']) / n:.1%}"
              f"   p̂ = {p_hat['value']:.3f} [{p_hat['ci_lo']:.3f} ; {p_hat['ci_hi']:.3f}]"
              f"   rappel du petit objet {summary['recall']['value']:.1%}")
        rail = summary.get('rail')      # mesure EXPLORATOIRE (DECISIONS.md, 17 nov.) : aucun verdict n'en depend
        if rail:
            cr = rail['counts']
            print(f"     au rail de la bifurcation (exploratoire) : flou {cr['floue'] / n:.1%}"
                  f"   autre {cr['autre'] / n:.1%}   nettes {(cr['nette-A'] + cr['nette-B']) / n:.1%}"
                  f"   p̂ = {rail['p_hat']['value']:.3f}")
    return h0b_passed if family == 'regression' else None


try:
    if STAGE == 'first_run':
        curve_args, runs_to_report = ['--run-id', FIRST_RUN], [FIRST_RUN]
    else:
        curve_args = ['--all']
        runs_to_report = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob('runs/*/eval.json'))
    curves = subprocess.run(['python', 'scripts/training_curves.py', *curve_args], capture_output=True, text=True)
    print(curves.stdout, curves.stderr)
    no_overfit_alert = curves.returncode == 0

    from IPython.display import Image, display
    for figure_path in sorted(glob.glob('runs/*/fig_training.png')):
        if STAGE != 'first_run' or FIRST_RUN in figure_path:
            print(figure_path)
            display(Image(figure_path))

    h0b_results = {run_id: report_run(run_id) for run_id in runs_to_report}
    if STAGE == 'first_run':
        h0b_ok = h0b_results.get(FIRST_RUN)
        print()
        if h0b_ok is None:
            print("=> run pas encore évalué : relancer la même session (STAGE = 'first_run').")
        elif h0b_ok and no_overfit_alert:
            print("=> FEU VERT : pas de surapprentissage détecté et H0b réussi. "
                  "Session suivante : STAGE = 'core_control'.")
        else:
            print("=> NE PAS lancer le cœur : "
                  + ("alerte de surapprentissage ; " if not no_overfit_alert else "")
                  + ("H0b échoué ; " if not h0b_ok else "")
                  + "comparer à J6 (docs/JOURNAL.md) et trancher avant la suite.")
except Exception as e:
    print(f"contrôle impossible ({type(e).__name__}: {e}) : la sauvegarde qui suit a quand même lieu")

## Sauvegarde de la session

Toujours exécutée, même si une cellule précédente a échoué à l'intérieur de son `try`.

In [ ]:
save_state(f"STAGE = {STAGE}")
!python scripts/run_plan.py --steps {STEPS} --groups core control --dry    # état en fin de session
n_evaluated_end = len(glob.glob('runs/*/eval.json'))
print(f"\n>>> RUNS ÉVALUÉS EN FIN DE SESSION : {n_evaluated_end}  ->  mettre MIN_EVALUATED = {n_evaluated_end} "
      "à la prochaine session, avec l'entrée flou-wm-etat sur sa dernière version.")

## 8. Analyse (quand les 12 runs cœur + contrôles sont « évalué »)

Verdicts H0 à H3, tables, figures, rapport. Avant, la plupart des verdicts seraient « non évaluables » (H1 à H3 comparent
3 familles sur 3 graines). Puis, à la main : la relecture de 100 cas et le test A/B (voir `docs/EXPERIMENTS.md`).

In [ ]:
if RUN_ANALYSIS:
    !python scripts/analyze.py
    try:
        from IPython.display import Markdown, Image, display
        display(Markdown(open('reports/report.md', encoding='utf-8').read()))
        for figure_path in sorted(glob.glob('reports/fig_*.png')):
            print(figure_path)
            display(Image(figure_path))
    except Exception as e:
        print("affichage du rapport impossible :", e)
else:
    print("analyse sautée (RUN_ANALYSIS = False)")

### Bandes d'images : ce que le modèle imagine

Illustrations seulement, aucun verdict n'en dépend.

In [ ]:
if RUN_ANALYSIS:
    if os.path.exists('runs/core-regression-p0.30-s0/final.pt'):
        !python scripts/filmstrip.py --run-id core-regression-p0.30-s0 --gif
    if os.path.exists('runs/core-diffusion-p0.30-s0/final.pt'):
        !python scripts/filmstrip.py --run-id core-diffusion-p0.30-s0 --n-steps 3 --gif
    try:
        from IPython.display import Image, display
        for figure_path in sorted(glob.glob('runs/core-*/filmstrip_*.png')):
            print(figure_path)
            display(Image(figure_path))
    except Exception as e:
        print("affichage impossible :", e)

## 9. Extension « décision » (facultatif) : le flou change-t-il une décision ?

Hypothèses H4 et H5, guidées par PETS (arXiv:1805.12114), PlaNet (arXiv:1811.04551) et Objective Mismatch (arXiv:2002.04523) :
voir `docs/EXTENSION_DECISION.md`. Après l'analyse, sur les runs évalués.

In [ ]:
if RUN_DECISION_L1:
    # H0f aux réglages du niveau 2 (DECISIONS.md, C7 révisé) : 32 suites, 20 parties ; puis le niveau 1, rapide
    !python scripts/validate_decision.py --games 20 --candidates 32
    !python scripts/evaluate_decision.py --all
else:
    print("extension niveau 1 sautée (RUN_DECISION_L1 = False)")

In [ ]:
# Niveau 2 (planification dans le modèle) : LONG. Estimation du test de fumée sur T4 aux réglages par défaut :
# régression ~1 h, diffusion ~12 h, jetons ~66-75 h PAR RUN ; avec --games 20 --candidates 32, environ 5 fois moins
# (jetons ~15 h par run : plus qu'une session Kaggle). À décider avant de l'activer (DECISIONS.md, C7).
# Relancer après une coupure : les runs déjà faits sont sautés.
if RUN_DECISION_L2:
    !python scripts/evaluate_decision.py --all --level 2 --games 4 --game-steps 10 --no-save   # chronométrage seulement
    !python scripts/evaluate_decision.py --all --level 2 --games 20 --candidates 32
    !python scripts/analyze.py   # le rapport contient maintenant H4 et H5
else:
    print("extension niveau 2 sautée (RUN_DECISION_L2 = False)")

## Sauvegarde des résultats d'analyse et d'extension

In [ ]:
if RUN_ANALYSIS or RUN_DECISION_L1 or RUN_DECISION_L2:
    save_state("analyse / extension")
else:
    print("rien de nouveau depuis la sauvegarde de session")